In [1]:
import numpy as np
import pandas as pd
import os, re, csv
from dotenv import load_dotenv

import warnings
warnings.filterwarnings('ignore')

load_dotenv()

True

In [2]:
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk import word_tokenize, sent_tokenize
import string
from string import punctuation

In [3]:
def cleanData(text, stemming = False, lemmatize=False):    
    text = text.lower().split()
    text = " ".join(text)
    text = re.sub(r"[^A-Za-z0-9^,!.\/'+\-=]", " ", text)
    text = re.sub(r"what's", "what is ", text)
    text = re.sub(r"\'s", " ", text)
    text = re.sub(r"\'ve", " have ", text)
    text = re.sub(r"can't", "cannot ", text)
    text = re.sub(r"n't", " not ", text)
    text = re.sub(r"i'm", "i am ", text)
    text = re.sub(r"\'re", " are ", text)
    text = re.sub(r"\'d", " would ", text)
    text = re.sub(r"\'ll", " will ", text)
    text = re.sub(r",", " ", text)
    text = re.sub(r"\.", " ", text)
    text = re.sub(r"!", " ! ", text)
    text = re.sub(r"\/", " ", text)
    text = re.sub(r"\^", " ^ ", text)
    text = re.sub(r"\+", " + ", text)
    text = re.sub(r"\-", " - ", text)
    text = re.sub(r"\=", " = ", text)
    text = re.sub(r"'", " ", text)
    text = re.sub(r"(\d+)(k)", r"\g<1>000", text)
    text = re.sub(r":", " : ", text)
    text = re.sub(r" e g ", " eg ", text)
    text = re.sub(r" b g ", " bg ", text)
    text = re.sub(r" u s ", " american ", text)
    text = re.sub(r"\0s", "0", text)
    text = re.sub(r" 9 11 ", "911", text)
    text = re.sub(r"e - mail", "email", text)
    text = re.sub(r"j k", "jk", text)
    text = re.sub(r"\s{2,}", " ", text)
    text = re.sub(r'\s+', ' ', text).strip()
    if stemming:
        st = PorterStemmer()
        txt = " ".join([st.stem(w) for w in text.split()])
    if lemmatize:
        wordnet_lemmatizer = WordNetLemmatizer()
        txt = " ".join([wordnet_lemmatizer.lemmatize(w) for w in text.split()])
    return text

In [4]:
special_character_removal=re.compile(r'[^a-z\d ]',re.IGNORECASE)
replace_numbers=re.compile(r'\d+',re.IGNORECASE)

def text_to_wordlist(text, remove_stopwords=False, stem_words=False):
    text = text.lower().split()
    
    text = " ".join(text)
    
    text=special_character_removal.sub('',text)
    text=replace_numbers.sub('',text)
    
    return(text)

In [5]:
train_df = pd.read_csv(os.getenv('train_csv'))
test_df = pd.read_csv(os.getenv('test_csv'))

train_df['comment_text'] = train_df['comment_text'].map(lambda x: cleanData(x))
test_df['comment_text'] = test_df['comment_text'].map(lambda x: cleanData(x))

In [6]:
classes = ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]

comments_train = train_df['comment_text'].values
comments_test = test_df['comment_text'].values

In [7]:
sentences_train=[]
for text in comments_train:
    sentences_train.append(text_to_wordlist(text))

sentences_test=[]
for text in comments_test:
    sentences_test.append(text_to_wordlist(text))

In [8]:
sentences_train[1]

'd aww  he matches this background colour i am seemingly stuck with thanks talk   january   utc'

In [9]:
import tensorflow
from tensorflow import keras
from tensorflow.keras.preprocessing.text import Tokenizer
from keras.preprocessing.sequence import pad_sequences
from keras.layers import *
from keras.models import Model
from tensorflow.keras.layers import BatchNormalization

In [10]:
MAX_SEQUENCE_LENGTH = 100
MAX_NB_WORDS = 25000
EMBEDDING_DIM = 64
VALIDATION_SPLIT = 0.1

In [11]:
tokenizer =  Tokenizer(num_words=MAX_NB_WORDS)
tokenizer.fit_on_texts(sentences_train)

In [12]:
seq_train = tokenizer.texts_to_sequences(sentences_train)
seq_test = tokenizer.texts_to_sequences(sentences_test)

X = pad_sequences(
    seq_train,
    maxlen=MAX_SEQUENCE_LENGTH,
    padding='post',
    truncating='post'
)

X_test = pad_sequences(
    seq_test,
    maxlen=MAX_SEQUENCE_LENGTH,
    padding='post',
    truncating='post'
)

y = train_df[classes]

In [13]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=VALIDATION_SPLIT, random_state=68
)

In [14]:
import pickle

with open('tokenizer.pkl','wb') as f:
    pickle.dump(tokenizer,f)

In [15]:
print("Training Shape :", X_train.shape)
print("Validation Shape :", X_val.shape)
print("Test Shape :", X_test.shape)

print("Training Labels :", y_train.shape)
print("Validation Labels :", y_val.shape)

Training Shape : (143613, 100)
Validation Shape : (15958, 100)
Test Shape : (153164, 100)
Training Labels : (143613, 6)
Validation Labels : (15958, 6)


In [16]:
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import Embedding, Dense, Dropout
from tensorflow.keras.layers import SimpleRNN, LSTM, Bidirectional

from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

In [17]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=2,
    restore_best_weights=True
)

checkpoint = ModelCheckpoint(
    'working_model.keras',
    monitor='val_loss',
    save_best_only=True
)

In [17]:
# SIMPLE RNN
model_rnn = Sequential()
model_rnn.add(
    Embedding(
        input_dim=MAX_NB_WORDS,
        output_dim=EMBEDDING_DIM,
        input_length=MAX_SEQUENCE_LENGTH
    )
)
model_rnn.add(SimpleRNN(64))
model_rnn.add(Dropout(0.3))
model_rnn.add(Dense(32, activation='relu'))
model_rnn.add(Dropout(0.3))
model_rnn.add(Dense(6, activation='sigmoid'))

model_rnn.compile(optimizer='adam', loss='binary_crossentropy',metrics=['accuracy'])

In [18]:
history_rnn = model_rnn.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=128,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/10
1122/1122 ━━━━━━━━━━━━━━━━━━━━ 157s 138ms/step - accuracy: 0.7398 - loss: 0.1577 - val_accuracy: 0.9937 - val_loss: 0.1282
Epoch 2/10
1122/1122 ━━━━━━━━━━━━━━━━━━━━ 160s 143ms/step - accuracy: 0.9653 - loss: 0.1271 - val_accuracy: 0.9936 - val_loss: 0.1147
Epoch 3/10
1122/1122 ━━━━━━━━━━━━━━━━━━━━ 156s 139ms/step - accuracy: 0.9860 - loss: 0.1212 - val_accuracy: 0.9937 - val_loss: 0.1166
Epoch 4/10
1122/1122 ━━━━━━━━━━━━━━━━━━━━ 156s 139ms/step - accuracy: 0.9936 - loss: 0.1256 - val_accuracy: 0.9937 - val_loss: 0.1397


In [19]:
model_rnn.save("simple_rnn.keras")

In [18]:
model_lstm = Sequential()

model_lstm.add(
    Embedding(
        input_dim=25000,
        output_dim=64,
        input_length=MAX_SEQUENCE_LENGTH
    )
)

model_lstm.add(LSTM(32))
model_lstm.add(Dropout(0.3))
model_lstm.add(Dense(16, activation="relu"))
model_lstm.add(Dropout(0.2))
model_lstm.add(Dense(6, activation="sigmoid"))

model_lstm.compile(optimizer="adam",loss="binary_crossentropy",metrics=["accuracy"])

In [ ]:
history_lstm = model_lstm.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=5,
    batch_size=128,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/5
1122/1122 ━━━━━━━━━━━━━━━━━━━━ 74s 64ms/step - accuracy: 0.7038 - loss: 0.1364 - val_accuracy: 0.9937 - val_loss: 0.0650
Epoch 2/5
1122/1122 ━━━━━━━━━━━━━━━━━━━━ 67s 60ms/step - accuracy: 0.9719 - loss: 0.0571 - val_accuracy: 0.9937 - val_loss: 0.0521
Epoch 3/5
1122/1122 ━━━━━━━━━━━━━━━━━━━━ 73s 65ms/step - accuracy: 0.9905 - loss: 0.0479 - val_accuracy: 0.9937 - val_loss: 0.0522
Epoch 4/5
 947/1122 ━━━━━━━━━━━━━━━━━━━━ 11s 65ms/step - accuracy: 0.9924 - loss: 0.0441

In [22]:
model_lstm.save("bilstm_mini.keras")

In [19]:
model_bilstm = Sequential()

model_bilstm.add(
    Embedding(
        input_dim=MAX_NB_WORDS,
        output_dim=EMBEDDING_DIM,
        input_length=MAX_SEQUENCE_LENGTH
    )
)

model_bilstm.add(Bidirectional(LSTM(256)))
model_bilstm.add(Dropout(0.3))
model_bilstm.add(Dense(128, activation="relu"))
model_bilstm.add(Dropout(0.2))
model_bilstm.add(Dense(64, activation="relu"))
model_bilstm.add(Dropout(0.1))
model_bilstm.add(Dense(6, activation="sigmoid"))

model_bilstm.compile(optimizer="adam",loss="binary_crossentropy",metrics=["accuracy"])

In [20]:
history_bilstm = model_bilstm.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=5,
    batch_size=128,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/5
1122/1122 ━━━━━━━━━━━━━━━━━━━━ 1271s 1s/step - accuracy: 0.8946 - loss: 0.0756 - val_accuracy: 0.9937 - val_loss: 0.0506
Epoch 2/5
1122/1122 ━━━━━━━━━━━━━━━━━━━━ 1447s 1s/step - accuracy: 0.9847 - loss: 0.0467 - val_accuracy: 0.9937 - val_loss: 0.0491
Epoch 3/5
1122/1122 ━━━━━━━━━━━━━━━━━━━━ 1450s 1s/step - accuracy: 0.9891 - loss: 0.0405 - val_accuracy: 0.9937 - val_loss: 0.0481
Epoch 4/5
1122/1122 ━━━━━━━━━━━━━━━━━━━━ 1376s 1s/step - accuracy: 0.9804 - loss: 0.0359 - val_accuracy: 0.9936 - val_loss: 0.0518
Epoch 5/5
1122/1122 ━━━━━━━━━━━━━━━━━━━━ 1373s 1s/step - accuracy: 0.9508 - loss: 0.0320 - val_accuracy: 0.9937 - val_loss: 0.0549


In [21]:
model_bilstm.save('bilstm2.keras')

In [22]:
test_pred = model_bilstm.predict(X_test)

4787/4787 ━━━━━━━━━━━━━━━━━━━━ 375s 78ms/step


In [23]:
test_pred

array([[9.9958938e-01, 5.0412440e-01, 9.7865516e-01, 1.0428949e-01,
        9.4307828e-01, 2.8807703e-01],
       [3.2832785e-04, 6.5649258e-10, 7.2662569e-06, 2.5297624e-07,
        1.9065739e-05, 3.7332558e-07],
       [1.0225210e-02, 8.8846127e-07, 5.7852664e-04, 4.2710068e-05,
        1.1065326e-03, 7.6024575e-05],
       ...,
       [8.8945038e-05, 2.9077378e-11, 1.2355298e-06, 2.6834984e-08,
        3.6439001e-06, 4.2322814e-08],
       [2.3240635e-04, 1.9290743e-10, 3.9302295e-06, 1.0564443e-07,
        1.0662362e-05, 1.7122522e-07],
       [9.8489684e-01, 1.8899377e-02, 8.9159304e-01, 2.4406097e-03,
        5.2969283e-01, 1.7401017e-02]], shape=(153164, 6), dtype=float32)

In [95]:
lstm_pred = model_lstm.predict(X_test) 

4787/4787 ━━━━━━━━━━━━━━━━━━━━ 116s 24ms/step


In [97]:
lstm_labels = (lstm_pred>=0.5).astype(int)
lstm_labels[0]

array([1, 1, 1, 1, 1, 0])

In [24]:
pred_labels = (test_pred >= 0.5).astype(int)

In [25]:
pred_labels[0]

array([1, 1, 1, 0, 1, 0])

In [26]:
sentences_test[0]

'yo bitch ja rule is more succesful then you will ever be whats up with you and hating you sad mofuckas i should bitch slap ur pethedic white faces and get you to kiss my ass you guys sicken me ja rule is about pride in da music man dont diss that shit on him and nothin is wrong bein like tupac he was a brother too fuckin white boys get things right next time'

In [27]:
classes

['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']

In [28]:
test_df['toxic'] = pred_labels[:,0]
test_df['severe_toxic'] = pred_labels[:,1]
test_df['obscene'] = pred_labels[:,2]
test_df['threat'] = pred_labels[:,3]
test_df['insult'] = pred_labels[:,4]
test_df['identity_hate'] = pred_labels[:,5]

In [29]:
test_df.to_csv('test_predictions.csv', index=False)

In [30]:
test_df

,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,00001cee341fdb12,yo bitch ja rule is more succesful then you wi...,1,1,1,0,1,0
1,0000247867823ef7,= = from rfc = = the title is fine as it is imo,0,0,0,0,0,0
2,00013b17ad220c46,= = sources = = zawe ashton on lapland,0,0,0,0,0,0
3,00017563c3f7919a,if you have a look back at the source the info...,0,0,0,0,0,0
4,00017695ad8997eb,i do not anonymously edit articles at all,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...
153159,fffcd0960ee309b5,i totally agree this stuff is nothing but too ...,1,0,0,0,0,0
153160,fffd7a9a6eb32c16,= = throw from out field to home plate = = doe...,0,0,0,0,0,0
153161,fffda9e8d6fafa9e,= = okinotorishima categories = = i see your c...,0,0,0,0,0,0
153162,fffe8f1340a79fc2,= = one of the founding nations of the eu - ge...,0,0,0,0,0,0


In [31]:
test_df.iloc[153163]['comment_text']

'stop already your bullshit is not welcome here i am no fool and if you think that kind of explination is enough well pity you'

In [32]:
train_df[train_df['identity_hate']==1]

,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate
42,001810bf8c45bf5f,you are gay or antisemmitian archangel white t...,1,0,1,0,1,1
105,00472b8e2d38d1ea,a pair of jew - hating weiner nazi schmucks,1,0,1,0,1,1
176,006b94add72ed61c,i think that your a fagget get a oife and burn...,1,0,1,1,1,1
218,008e0818dde894fb,kill all niggers i have hard that others have ...,1,0,1,0,1,1
238,0097dd5c29bf7a15,u r a tw fuck off u gay boy u r smelly fuck ur...,1,0,1,0,1,1
...,...,...,...,...,...,...,...,...
159281,fb726deec64157bd,lol ! ! you are gay ! ! you will never know ho...,1,1,1,0,1,1
159336,fc3efa2f6f025f6d,oh fuck off the pansy jew would just whine abo...,1,0,1,0,1,1
159400,fd052883fa6a8697,shalom semite get the fuck out of here i will ...,1,1,1,1,1,1
159449,fdce660ddcd6d7ca,i think he is a gay fag ! ! !,1,0,0,0,0,1


In [33]:
train_df.iloc[159400]['comment_text']

'shalom semite get the fuck out of here i will kill you you son of a bitch if you do not leave wikipedia if you tell anyone i said this to you i will rape your jew family choke on a bagel you faggoty hymie cock sucker you better fucking die before i hunt you down and piss on your face if you get me blocked for this i will find you and shoot you i just want your jew fucking self gone you dumb motherfucking piece of shit g - d damn jew die ! shalom we came in'